# 08. Dual-stream CrossDiT K6: два потока Audio + Video

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

**Результат:** по исходному коду восстановить порядок FusedTransformerDecoderBlock, источники Q и K/V обоих межмодальных attention и роль флагов, затем проверить wiring причинными опытами

**Перед началом:** K6 07; Q/K/V и RoPE из 06. [Порядок всех материалов](../../curriculum/CONTENTS.md).
[Курс K6](../README.md) · [Покомпонентная карта](../COMPONENT_GUIDE.md) · [Предыдущий опыт K5](../../kandinsky-5/notebooks/06_dit_and_flow_matching.ipynb).

Основной архитектурный источник: [Kandinsky 6 на Hugging Face Papers](https://huggingface.co/papers/2610.05608),
[полный отчёт, §3.2, Figures 1–2, Table 2](https://arxiv.org/html/2610.05608v1). Открывайте статью рядом с notebook.
Мы различаем опубликованную архитектуру, конкретный checkpoint config и результат нашего опыта.

**Код для сравнения (закреплённая revision `7ceeefad`):**
- [`core/components/dit.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/components/dit.py)
- [`core/components/piflow_dit.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/components/piflow_dit.py)
- [`core/algo/denoise_loop.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/algo/denoise_loop.py)
- [`core/algo/piflow_sampler.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/algo/piflow_sampler.py)

Run All выполняет небольшие CPU-опыты. Случайная уменьшенная сеть проверяет вычисления;
pretrained генерацию, качество голоса и производительность M3 она не измеряет.
Большие веса не скачиваются, ComfyUI jobs не отправляются. В Colab первая ячейка получает
исходники курса из GitHub, если их ещё нет; пакеты не устанавливает. На Mac используется существующий checkout.
Ниже объяснение идёт перед каждым опытом.


In [ ]:
from pathlib import Path
import json, sys, os, math
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = (parent, parent / 'kandinsky-6')
    found = next((p for p in candidates if (p / 'labkit/components.py').is_file()), None)
    if found is not None:
        ROOT = found.resolve()
        break
else:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Откройте notebook внутри checkout generative-ai-research.')
    import subprocess
    checkout=Path('/content/generative-ai-research-k6-components')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','--branch',
            os.environ.get('K6_LABS_REF','main'),
            'https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    ROOT=checkout/'kandinsky-6'
    if not (ROOT/'labkit/components.py').is_file():
        raise FileNotFoundError('В выбранной revision нет component labs; проверьте K6_LABS_REF.')
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'artifacts/component_labs/mpl-cache'))
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from labkit.components import (config, native, show_source, native_causal_conv,
    tiny_model, tiny_inputs, activate_gates, fresh_forward, write_metrics, figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
np.random.seed(42)
LAB = '08_dual_stream_audio_video_crossdit'
metrics = {}
print('CPU teaching mode; PyTorch', torch.__version__)


## 0. Вопрос, вход и выход

**Вопрос:** как два latent потока разной ширины и частоты обмениваются информацией внутри одного блока,
и какое время управляет каждым обменом?

«Dual-stream» означает **две модальности**, а не stereo audio. Вход одного `FusedTransformerDecoderBlock`:

| Аргумент | Форма | Смысл |
|---|---|---|
| `vis` | `[B, N_v, D_v]` | video tokens после patch embedding или предыдущего блока |
| `aud` | `[B, N_a, D_a]` | audio tokens: по одному на audio latent frame, без spatial patching |
| `text_v`, `text_a` | `[B, L, D_v]`, `[B, L, D_a]` | выходы **двух разных** text refiners из одного prompt |
| `time_embed` | `(c_v, c_a)`, каждый `[B, time_dim]` | time + pooled для каждой модальности; t_v и t_a могут различаться |
| `vis_rope`, `aud_rope` | rotations | 3D RoPE для video, 1D RoPE для audio |

Выход: `(vis', aud')` тех же форм. Основное чтение: [K6 §3.2, Figures 1–2](https://arxiv.org/html/2610.05608v1#S3.SS2),
[страница статьи](https://huggingface.co/papers/2610.05608). Статья задаёт порядок sub-layers; точный источник K/V
и роль флагов — детали pinned кода, которые проверяем ниже.


In [ ]:
D=native();tensors=native('core.tensors');tr=json.loads((ROOT/'reference/hf/Kandinsky-6.0-Lite-5s-Diffusers/transformer/config.json').read_text())
hd_v,hd_a=sum(tr['axes_dims']),sum(tr['axes_dims_a'])
Nv=31*30*54;Na=native('core.algo.prepare_latents').audio_latent_duration(31)
with torch.device('meta'):
    big=D.FusedTransformerDecoderBlock(tr['model_dim'],tr['time_dim'],tr['ff_dim'],hd_v,
        tr['model_dim_a'],tr['time_dim_a'],tr['ff_dim_a'],hd_a,engine='sdpa',text_token_padding=True,
        ca_rope=tr['ca_rope'],cross_gates=tr['cross_gates'],fix_modulation=tr['fix_modulation'])
count=lambda *mods:sum(p.numel() for m in mods for p in m.parameters())
per_block={'video stream (SA, text CA, FFN, modulation)':count(big.video_dec_block),
           'audio stream (SA, text CA, FFN, modulation)':count(big.audio_dec_block),
           'fusion attention (va + av)':count(big.va_cross_attention,big.av_cross_attention),
           'fusion modulation + norms':count(big.va_modulation,big.av_modulation)}
lite_streams={'D_v / heads':[tr['model_dim'],tr['model_dim']//hd_v],'D_a / heads':[tr['model_dim_a'],tr['model_dim_a']//hd_a],
              'head_dim v/a':[hd_v,hd_a],'ffn v/a':[tr['ff_dim'],tr['ff_dim_a']],'fused blocks':tr['num_visual_blocks'],
              'N_v (121 frames, 480x864)':Nv,'N_a (same clip)':Na,
              'flags':{k:tr[k] for k in ['ca_rope','cross_gates','fix_modulation']},
              'params per block, M':{k:round(v/1e6,1) for k,v in per_block.items()},
              'all fused blocks, B':round(sum(per_block.values())*tr['num_visual_blocks']/1e9,2)}
metrics['lite_streams']=lite_streams;print(json.dumps(lite_streams,indent=2))


Асимметрия видна в числах: audio stream имеет половину width и FFN video stream и в сотни раз меньше tokens.
Attention между потоками прямоугольная, $N_v\times N_a$; одного audio token на video token не требуется.

## 1. Направление определяется по query

| Статья (Fig. 1) | Атрибут кода | Q (получатель) | K/V (источник) | Что обновляется |
|---|---|---|---|---|
| V2A block: video queries attend to audio keys/values | `va_cross_attention` | video | audio | video |
| A2V block: audio queries attend to video keys/values | `av_cross_attention` | audio | video | audio |

Имя «V2A» легко прочитать как «video → audio», хотя этот блок обновляет **video**. Поэтому направление проверяем по Q:
число выходных строк равно числу queries, а K/V проекции переводят ширину источника в ширину получателя.


In [ ]:
v_from_a=D.MultiheadCrossAttention(48,8,32,engine='sdpa')
a_from_v=D.MultiheadCrossAttention(32,8,48,engine='sdpa')
v=torch.randn(1,8,48);a=torch.randn(1,6,32)
with torch.no_grad():dv=v_from_a(v,a);da=a_from_v(a,v)
assert dv.shape==v.shape and da.shape==a.shape
metrics['fusion_projection_shapes']={
    'video_Q [out,in]':list(v_from_a.to_query.weight.shape),'audio_K_to_video [out,in]':list(v_from_a.to_key.weight.shape),
    'audio_Q [out,in]':list(a_from_v.to_query.weight.shape),'video_K_to_audio [out,in]':list(a_from_v.to_key.weight.shape)}
print(json.dumps(metrics['fusion_projection_shapes'],indent=2))


## 2. Точный порядок внутри `FusedTransformerDecoderBlock`

Обозначим $\mathrm{M}(h;b,s)=(1+s)\,\mathrm{LN}(h)+b$; $c_v,c_a$ — time embeddings потоков.

**Video, часть 1.** Self-attention с 3D RoPE: $V\leftarrow V+g^v_{sa}\odot \mathrm{SA}_v(\mathrm{M}(V))$.
Затем $\tilde V=\mathrm{M}(V;b^v_{ca},s^v_{ca})$ — сохранённый `vis_pre_ca` — и $T_v=\mathrm{CA}_v(\tilde V,\,\text{text}_v)$.
Text update video **вычислен, но ещё не применён**.

**Audio.** $A\leftarrow A+g^a_{sa}\odot\mathrm{SA}_a(\mathrm{M}(A))$; $\tilde A$ = `aud_pre_ca`; сразу
$A\leftarrow A+g^a_{ca}\odot \mathrm{CA}_a(\tilde A,\text{text}_a)$.

**Fusion.**

1. $V\leftarrow V+g^v_{ca}\odot T_v$ — теперь text update применён к video.
2. Query states: $\hat V=\mathrm{M}(V;b_{va},s_{va})$, $\hat A=\mathrm{M}(A;b_{av},s_{av})$ — **после** text update.
3. $\Delta V=\mathrm{Attn}(Q=\hat V,\;K,V=\tilde A)$ и $\Delta A=\mathrm{Attn}(Q=\hat A,\;K,V=\tilde V)$ — K/V **до** text update.
   Оба выхода вычисляются до применения любого из них.
4. $V\leftarrow V+g_{\to v}\odot\Delta V$, $A\leftarrow A+g_{\to a}\odot\Delta A$; затем отдельные FFN.

Q/K каждой attention проходят RMSNorm по head; при `ca_rope=True` video side получает 3D, audio side — 1D rotations.
Схема ниже рисует именно это: K/V берутся из `vis_pre_ca`/`aud_pre_ca`, Q — из состояния после text update.
Затем — полное тело блока из pinned source и ручная сборка по этой записи.


In [ ]:
from matplotlib.patches import FancyBboxPatch
fig,ax=plt.subplots(figsize=(12,4.6));ax.set_xlim(0,12.2);ax.set_ylim(0,4.9);ax.axis('off')
boxes=[('Self-attn',.2,1.6),('M(·): pre-CA\nsaved state',2.25,1.7),('Text CA\n(cross-attn)',4.45,1.6),
       ('text residual,\nthen M(·): Q',6.55,1.6),('Cross-modal\nattention',8.65,1.6),('FFN',10.75,1.2)]
rows={'video':(3.3,'#d8eafa','VIDEO  D_v=1792, 28 heads, 3D RoPE'),'audio':(.9,'#fbe3cc','AUDIO  D_a=896, 14 heads, 1D RoPE')}
for name,(y,color,label) in rows.items():
    ax.text(.2,y+1.0,label,fontsize=10,weight='bold')
    for i,(title,x,w) in enumerate(boxes):
        ax.add_patch(FancyBboxPatch((x,y),w,.75,boxstyle='round,pad=.06',facecolor=color,edgecolor='#555555'))
        ax.text(x+w/2,y+.37,title,ha='center',va='center',fontsize=8.5)
        if i<len(boxes)-1:ax.annotate('',xy=(boxes[i+1][1],y+.37),xytext=(x+w,y+.37),arrowprops={'arrowstyle':'->'})
ax.annotate('',xy=(9.2,1.65),xytext=(3.1,3.3),arrowprops={'arrowstyle':'->','color':'#2f6db0','lw':1.6})
ax.annotate('',xy=(9.2,3.3),xytext=(3.1,1.65),arrowprops={'arrowstyle':'->','color':'#b0662f','lw':1.6})
ax.plot([],[],color='#2f6db0',label='vis_pre_ca → K/V of audio queries')
ax.plot([],[],color='#b0662f',label='aud_pre_ca → K/V of video queries')
ax.legend(loc='center right',fontsize=9,frameon=False)
ax.text(.2,.15,'Q comes from the destination stream after its text update; K/V come from the other stream before it. '
        'Both cross outputs are computed before either is added.',fontsize=9)
fig.tight_layout();fig.savefig(figure_path(LAB,'dual_stream_architecture'));plt.show()


In [ ]:
show_source('core/components/dit.py','FusedTransformerDecoderBlock')


In [ ]:
model=tiny_model().eval();block=model.visual_transformer_blocks[0];activate_gates(block)
Dv,Da=block.model_dim,block.model_dim_a
vr=model.visual_rope((2,2,2),[torch.arange(2)]*3).flatten(0,2);ar=model.audio_rope(torch.arange(6))
v=torch.randn(1,8,Dv);a=torch.randn(1,6,Da);tv=torch.randn(1,16);ta=torch.randn(1,16)
text_v=torch.randn(1,5,Dv);text_a=torch.randn(1,5,Da)
M,G=tensors.apply_scale_shift_norm,tensors.apply_gate_sum
def manual_block(vis,aud,variant='source'):
    vb,ab=block.video_dec_block,block.audio_dec_block
    sa,ca,ff=vb.visual_modulation(tv).chunk(3,-1)
    b,s,g=sa.chunk(3,-1);vis=G(vis,vb.self_attention(M(vb.self_attention_norm,vis,s,b),vr),g)
    b_v,s_v,g_v=ca.chunk(3,-1);vis_pre=M(vb.cross_attention_norm,vis,s_v,b_v);text_out=vb.cross_attention(vis_pre,text_v)
    sa,ca,ff_a=ab.visual_modulation(ta).chunk(3,-1)
    b,s,g=sa.chunk(3,-1);aud=G(aud,ab.self_attention(M(ab.self_attention_norm,aud,s,b),ar),g)
    b,s,g=ca.chunk(3,-1);aud_pre=M(ab.cross_attention_norm,aud,s,b);aud=G(aud,ab.cross_attention(aud_pre,text_a),g)
    vis=G(vis,text_out,g_v)
    # Released flags: fix_modulation=True -> destination time for shift/scale; cross_gates=True -> gates swapped.
    va_b,va_s,va_g=block.va_modulation(tv).split([Dv,Dv,Da],-1)
    av_b,av_s,av_g=block.av_modulation(ta).split([Da,Da,Dv],-1)
    q_v=M(block.va_normalization,vis,va_s,va_b);q_a=M(block.av_normalization,aud,av_s,av_b)
    kv_a,kv_v=(q_a,q_v) if variant=='post-text K/V' else (aud_pre,vis_pre)
    dv=block.va_cross_attention(q_v,kv_a,rope_q=vr,rope_kv=ar)
    if variant=='sequential V then A':
        vis=G(vis,dv,av_g);kv_v=M(vb.cross_attention_norm,vis,s_v,b_v)
    da=block.av_cross_attention(q_a,kv_v,rope_q=ar,rope_kv=vr)
    if variant!='sequential V then A':vis=G(vis,dv,av_g)
    aud=G(aud,da,va_g)
    b,s,g=ff.chunk(3,-1);vis=G(vis,vb.feed_forward(M(vb.feed_forward_norm,vis,s,b)),g)
    b,s,g=ff_a.chunk(3,-1);aud=G(aud,ab.feed_forward(M(ab.feed_forward_norm,aud,s,b)),g)
    return vis,aud
with torch.no_grad():
    nv,na=block(v,a,text_v,text_a,(tv,ta),vr,ar)
    order_rows={}
    for variant in ['source','post-text K/V','sequential V then A']:
        mv,ma=manual_block(v,a,variant)
        order_rows[variant]={'video max error':(mv-nv).abs().max().item(),'audio max error':(ma-na).abs().max().item()}
assert max(order_rows['source'].values())<1e-5
assert max(order_rows['post-text K/V'].values())>1e-4 and order_rows['sequential V then A']['audio max error']>1e-4
metrics['manual_block_vs_native']=order_rows;print(json.dumps(order_rows,indent=2))


Ручная сборка по записи выше совпадает с native forward до округления float32. Две правдоподобные, но неверные схемы
дают заметную ошибку: «K/V после text update» и «сначала V, затем A из обновлённого V».
Схема из статьи, «self → text → cross-modal → FFN», верна для residual updates, но не говорит, из какого состояния берутся K/V.

## 3. Флаги: какое время управляет fusion

`va_modulation` строит shift/scale для video queries, `av_modulation` — для audio queries.
`fix_modulation` выбирает, **чьё время** подаётся в эти modulations. `cross_gates` меняет split:
video-sized gate берётся из `av_modulation`, audio-sized — из `va_modulation`.

| `fix_modulation` | `cross_gates` | shift/scale video query | gate video update | shift/scale audio query | gate audio update |
|---|---|---|---|---|---|
| False | False | $c_a$ | $c_a$ | $c_v$ | $c_v$ |
| False | True | $c_a$ | $c_v$ | $c_v$ | $c_a$ |
| True | False | $c_v$ | $c_v$ | $c_a$ | $c_a$ |
| **True** | **True** (Lite и Pro configs) | $c_v$ | $c_a$ | $c_a$ | $c_v$ |

В released configs нормализация query управляется временем получателя, а gate его обновления — временем **источника**.
Ветка `fix_modulation=False` подаёт $c_a$ в `va_modulation`, созданную с `time_dim`; это работает потому, что `time_dim == time_dim_a`.
Таблицу проверяем hooks: какое время реально получает каждая modulation и из какой modulation приходит gate каждого потока.


In [ ]:
def routing(fix,cross):
    b=D.FusedTransformerDecoderBlock(48,16,96,8,32,16,64,8,engine='sdpa',ca_rope=True,cross_gates=cross,fix_modulation=fix)
    activate_gates(b);seen={}
    def tag(name):
        def hook(module,inputs,output):
            seen[name]='c_v' if inputs[0] is tv else 'c_a' if inputs[0] is ta else '?'
        return hook
    hs=[b.va_modulation.register_forward_hook(tag('va')),b.av_modulation.register_forward_hook(tag('av'))]
    with torch.no_grad():b(v,a,text_v,text_a,(tv,ta),vr,ar)
    for h in hs:h.remove()
    gate_v=seen['av'] if cross else seen['va'];gate_a=seen['va'] if cross else seen['av']
    return {'video query':seen['va'],'video gate':gate_v,'audio query':seen['av'],'audio gate':gate_a}
flag_rows={f'fix={fix}, cross={cross}':routing(fix,cross) for fix in (False,True) for cross in (False,True)}
assert flag_rows['fix=True, cross=True']=={'video query':'c_v','video gate':'c_a','audio query':'c_a','audio gate':'c_v'}
metrics['fusion_time_routing']=flag_rows;print(json.dumps(flag_rows,indent=2))


Gate source для `cross=True` выводится из split в коде; его правильность для released флагов уже подтвердила ручная сборка в разделе 2.

## 4. Zero-init invariance, затем причинное отключение связей

Свежий блок — тождественное отображение: все gates нулевые. Затем включим малые случайные modulation weights,
чтобы связи стали наблюдаемыми. Это исследование функции, а не синхронизации pretrained звука.

Предсказание: изменение A влияет на V при открытой ветви video-query; при `va_gate_scale=0` это влияние исчезает **точно**,
потому что внутри одного блока другого пути A→V нет. Аналогично для V→A и `av_gate_scale`.
Эти аргументы — probe-параметры блока: `DiffusionTransformer3D.forward` их не передаёт, в pipeline они всегда равны 1.


In [ ]:
fresh_block=tiny_model().eval().visual_transformer_blocks[0]
with torch.no_grad():v0,a0=fresh_block(v,a,text_v,text_a,(tv,ta),vr,ar)
identity=max((v0-v).abs().max().item(),(a0-a).abs().max().item());assert identity==0
def run(vv,aa,va=1.,av=1.):
    return block(vv,aa,text_v,text_a,(tv,ta),vr,ar,va_gate_scale=va,av_gate_scale=av)
perturb_a=torch.randn_like(a);perturb_v=torch.randn_like(v)
with torch.no_grad():
    base_v,base_a=run(v,a)
    changed_v,_=run(v,a+perturb_a)
    _,changed_a=run(v+perturb_v,a)
    off_v,_=run(v,a,va=0);off_v2,_=run(v,a+perturb_a,va=0)
    _,off_a=run(v,a,av=0);_,off_a2=run(v+perturb_v,a,av=0)
rows={'A affects V: active':(changed_v-base_v).abs().max().item(),
      'A affects V: va_gate_scale=0':(off_v2-off_v).abs().max().item(),
      'V affects A: active':(changed_a-base_a).abs().max().item(),
      'V affects A: av_gate_scale=0':(off_a2-off_a).abs().max().item()}
assert rows['A affects V: active']>0 and rows['V affects A: active']>0
assert rows['A affects V: va_gate_scale=0']==0 and rows['V affects A: av_gate_scale=0']==0
metrics['zero_gate_identity_error']=identity;metrics['directional_perturbations']=rows
fig,ax=plt.subplots(figsize=(8,3));ax.barh(list(rows),[max(x,1e-12) for x in rows.values()]);ax.set_xscale('log')
ax.set_xlabel('Max |Δ output| after perturbing the other stream (log; exact 0 drawn at 1e-12)')
ax.set_title('One tiny fused block, random weights: directional cuts')
fig.tight_layout();fig.savefig(figure_path(LAB,'cross_modal_ablation'));plt.show();print(rows)


## 5. Gradient показывает путь обучения, не качество синхронизации

Пусть scalar loss зависит только от audio output. При открытой ветви audio-query он даёт gradient на video input.
Поэтому совместное обучение отличается от двух независимых T2V/T2A. Но наличие gradient само по себе
не доказывает lip-sync или сходство голоса: для этого нужны pretrained веса и внешние метрики (SyncNet-подобные, SECS, WER).


In [ ]:
vg=v.clone().requires_grad_()
_,audio_on=run(vg,a)
g_on=torch.autograd.grad(audio_on.square().mean(),vg)[0]
_,audio_off=run(vg,a,av=0)
g_off=torch.autograd.grad(audio_off.square().mean(),vg)[0]
metrics['audio_loss_gradient_to_video_on']=g_on.norm().item()
metrics['audio_loss_gradient_to_video_off']=g_off.norm().item()
assert g_on.norm()>0 and g_off.norm()==0
print('gradient norms on/off:',g_on.norm().item(),g_off.norm().item())


## 6. Unimodal forward, joint forward, partial sampling и decode

| Режим | Вызов | Fusion | Время |
|---|---|---|---|
| video-only | `forward(x_video, x_audio=None)` | нет; text update video применяется в `else`-ветви | `video_*` refiner и $c_v$ |
| audio-only | `forward(None, x_audio)` | нет; video модули не читаются | `audio_*` refiner и $c_a$ |
| joint | оба latent | в каждом блоке | `time=[t_v, t_a]` могут различаться (статья §4.2.2: independent_time) |
| partial sampling | joint forward; известная модальность получает `timesteps[-1]·1000` = 0, т.е. чистые данные | да: чистая модальность — условие | Euler обновляет только генерируемый поток |
| π-Flow distilled | `PiFlowDiffusionTransformer3D` требует fused tuple | да, оба latent всегда | частичность задаётся time scale 0 в `piflow_sampler` |
| decode одного выхода | VAE/vocoder после sampling | не влияет на DiT | — |

Предсказание для первой строки: video-only forward **равен** joint forward, в котором закрыта ветвь video-query
во всех блоках. Audio влияет на video только через неё. Проверим на tiny native модели, установив `va_gate_scale=0`
через forward pre-hook. Это probe, а не опция pipeline.


In [ ]:
joint=tiny_model().eval();activate_gates(joint);_,_,jkw=tiny_inputs(joint)
def close_video_query(module,args,kwargs):
    kwargs['va_gate_scale']=0.;return args,kwargs
with torch.no_grad():
    joint_v,joint_a=fresh_forward(joint,jkw)
    video_only=fresh_forward(joint,dict(jkw,x_audio=None,audio_rope=None))
    hs=[b.register_forward_pre_hook(close_video_query,with_kwargs=True) for b in joint.visual_transformer_blocks]
    cut_v,_=fresh_forward(joint,jkw)
    for h in hs:h.remove()
    t_grid=native('core.algo.flow_matching').flow_match_timesteps(10,5.0,'cpu')
modes={'joint vs video-only, open fusion':(joint_v-video_only).abs().max().item(),
       'joint with video-query cut vs video-only':(cut_v-video_only).abs().max().item(),
       'frozen-modality model time (1000*t_final)':float(t_grid[-1]*1000),
       'joint output shapes':[list(joint_v.shape),list(joint_a.shape)]}
assert modes['joint vs video-only, open fusion']>1e-4 and modes['joint with video-query cut vs video-only']<1e-5
assert modes['frozen-modality model time (1000*t_final)']==0
metrics['forward_modes']=modes;print(json.dumps(modes,indent=2))


## 7. Почему эти опыты ещё не lip-sync, и с чем сравнивать K6

Cuts и gradients доказывают **wiring**: связь существует, направлена так, как записано, и закрывается указанным gate.
Они не говорят, использует ли обученная модель эту связь для синхронизации губ, ритма или тембра.
Для таких выводов нужны pretrained веса, held-out prompts и внешние метрики — см. [протокол исследования](../../curriculum/RESEARCH_PROTOCOL.md).

Сравнение с первичными работами (по их abstracts; детали проверяйте в статьях):

| Система | Потоки | Обмен между потоками |
|---|---|---|
| K6 | асимметричные: video $D_v$=1792, audio $D_a$=896 (Lite), одинаковое число блоков | bidirectional cross-attention в каждом блоке; K/V из pre-text-CA states; 3D/1D RoPE |
| [Ovi](https://arxiv.org/abs/2510.01284) | twin backbones с **идентичной** архитектурой | blockwise bidirectional cross-attention + scaled RoPE для timing |
| [LTX-2](https://arxiv.org/abs/2601.03233) | асимметричные: 14B video, 5B audio | bidirectional AV cross-attention с temporal positional embeddings; cross-modality AdaLN |

Общая идея — blockwise fusion; codec, text encoder, normalization и источник K/V у каждой системы свои.
Не считайте Ovi или LTX-2 буквальной реализацией K6, и не путайте LTX-2 с LTX-Video.

Необычная деталь, которую нужно сохранять при переносе: `OutLayerAudio` применяет LayerNorm дважды
(см. [05](05_nn_patches_and_modulation.ipynb), раздел 3). Комментарий в pinned source прямо называет это поведением
reference training. Без отдельного доказательства это контракт, а не bug.


## Самостоятельное исследование

1. Повторите ручную сборку раздела 2 для `fix_modulation=False, cross_gates=False`: какие строки `manual_block` нужно изменить? Совпадение с native — критерий.
2. Повторите directional ablation при `ca_rope=False` с 5 seeds; отчёт — медиана и разброс sensitivity, без вывода о lip-sync.
3. Переставьте audio tokens и их RoPE одновременно, затем только tokens. Какие инвариантности вы ожидаете для video output и почему?
4. Гипотеза для partial sampling: при известном чистом audio ($t_a=0$) gate audio→video в released конфигурации управляется $c_a$. Проверьте hook-ом в joint forward.
5. Для voice-FiLM гипотезы отметьте insertion point и проверьте путь gradient к адаптеру при frozen backbone.

Перед изменением запишите гипотезу. Сохраните baseline, меняйте один фактор и повторяйте по 3–5 seeds. Файл metrics.json содержит измерение конкретного запуска, а не статистическую оценку качества модели.


In [ ]:
write_metrics(LAB, metrics)


## Критерий понимания

Вы можете нарисовать оба потока одного блока, подписать для каждой attention Q, K/V, время shift/scale и время gate,
воспроизвести блок вручную и объяснить, почему directional cut подтверждает wiring, но не AV synchronization.
**Следующий шаг:** [09 — joint Flow Matching](09_flow_matching_training.ipynb), где эти потоки обучаются на общей velocity loss.

[Вернуться к маршруту K6](../README.md) · [Полное содержание](../../curriculum/CONTENTS.md) · [Шаблон отчёта](../../curriculum/EXPERIMENT_TEMPLATE.md).


[← 07. От DiT к CrossDiT: как denoiser устроен изнутри](07_dit_and_text_crossdit.ipynb) · [09. Joint Flow Matching: path, velocity и учебное обучение →](09_flow_matching_training.ipynb)
